In [33]:
import pandas as pd
from datasets import Dataset, DatasetDict

# data = {
#     "text": [
#         "Instruction: Segment the following feedback into a summary.\nInput: The attorney demonstrated excellent drafting skills in complex agreements but occasionally missed minor clauses related to confidentiality.\nOutput: The attorney drafted complex agreements well but sometimes overlooked confidentiality clauses.",

#         "Instruction: Segment the following feedback into a summary.\nInput: The paralegal showed initiative in gathering evidence and coordinating with witnesses, ensuring all materials were ready ahead of schedule.\nOutput: The paralegal proactively managed evidence and witness coordination ahead of schedule.",

#         "Instruction: Segment the following feedback into a summary.\nInput: The legal consultant provided thorough compliance reviews but failed to adapt quickly when regulations changed mid-project.\nOutput: The consultant was thorough in compliance reviews but slow to adjust to regulatory changes.",

#         "Instruction: Segment the following feedback into a summary.\nInput: The associate handled client communication effectively and resolved queries quickly but struggled with courtroom confidence.\nOutput: The associate communicated well with clients but lacked confidence in court.",

#         "Instruction: Segment the following feedback into a summary.\nInput: The firm’s billing process lacked transparency, though the quality of legal advice was consistently praised by clients.\nOutput: The firm’s legal advice was strong, but billing transparency needs improvement.",

#         "Instruction: Segment the following feedback into a summary.\nInput: The senior counsel provided insightful strategic guidance during arbitration but relied heavily on junior staff for documentation.\nOutput: The senior counsel offered strong strategic guidance but depended on juniors for documentation.",

#         "Instruction: Segment the following feedback into a summary.\nInput: The junior lawyer displayed strong research abilities yet failed to meet internal deadlines on multiple occasions.\nOutput: The junior lawyer excelled in research but struggled to meet deadlines.",

#         "Instruction: Segment the following feedback into a summary.\nInput: The contract review was comprehensive, and all risk factors were identified early, demonstrating strong analytical skills.\nOutput: The contract review was thorough and demonstrated strong analytical ability.",

#         "Instruction: Segment the following feedback into a summary.\nInput: The litigation team collaborated well under pressure and achieved a favorable outcome for the client despite tight timelines.\nOutput: The litigation team worked efficiently under pressure to secure a favorable outcome.",

#         "Instruction: Segment the following feedback into a summary.\nInput: The legal partner managed negotiations assertively, ensuring favorable contract terms for the client while maintaining professionalism.\nOutput: The partner negotiated assertively and secured favorable contract terms professionally."
#     ]
# }


# Transform the raw data into a structured list of dictionaries
# processed_data = []
# for entry in data['text']:
#     parts = entry.split("\n")
#     # Assuming a consistent format of "Instruction", "Input", "Output"
#     instruction = parts[0].replace("Instruction: ", "").strip()
#     input_text = parts[1].replace("Input: ", "").strip()
#     output_text = parts[2].replace("Output: ", "").strip()
#     processed_data.append({"instruction": instruction, "input": input_text, "output": output_text})

# # Convert to pandas DataFrame
# df = pd.DataFrame(processed_data)

import pandas as pd
from datasets import Dataset, DatasetDict

df = pd.read_json('data.json', lines=True).head(100)
df["instruction"]="Segment the following feedback into a summary."

# Convert to Hugging Face Dataset format
dataset = Dataset.from_pandas(df)

# Split the small dataset into train and test
dataset = dataset.train_test_split(test_size=0.2)
# For very small datasets like this, you might not even need a test set
# or just train on the whole thing and evaluate manually.
# For simplicity, we will split it.

# The dataset dictionary will contain 'train' and 'test' splits
# Create a DatasetDict for easier use with Trainer
dataset_dict = DatasetDict({
    'train': dataset['train'],
    'test': dataset['test']
})

print(dataset_dict)


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 80
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 20
    })
})


In [34]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

model_name = "t5-small"  # A smaller, faster model for demonstration
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


In [35]:
def preprocess_function(examples):
    inputs = [f"{examples['instruction']} {examples['input']}" for _ in examples['input']]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    labels = tokenizer(examples['output'], max_length=150, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(preprocess_function, batched=True)


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 1193.82 examples/s]


In [36]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 80
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 20
    })
})

In [37]:
import numpy as np
import evaluate
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# --- Step 5: Fine-tune the model ---
# For fine-tuning a sequence-to-sequence model like T5,
# use Seq2SeqTrainingArguments and Seq2SeqTrainer.

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-summarization-finetuned",  # Directory to save the model checkpoints
    eval_strategy="epoch",                # Evaluate after each epoch
    learning_rate=2e-5,
    per_device_train_batch_size=4,              # Adjust based on your GPU memory
    per_device_eval_batch_size=4,               # Adjust based on your GPU memory
    weight_decay=0.01,
    save_total_limit=3,                         # Keep only the last 3 checkpoints
    num_train_epochs=3,                         # Small number of epochs for demonstration
    fp16=True,                                  # Use mixed precision for faster training
    predict_with_generate=True,                 # Generate predictions for evaluation
    logging_steps=10,
    report_to="none"                            # Disable logging to services like Weights & Biases
)

# Load the ROUGE metric
rouge = evaluate.load("rouge")

# Define the compute_metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels as they were not processed by the model
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # ROUGE metric expects a list of predictions and a list of references
    # It also handles different types of ROUGE scores (1, 2, L)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    # Calculate the average length of predictions
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# Define the data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()





/var/folders/cf/1n82cp8j5477wf_t828mwrlh0000gn/T/ipykernel_1485/3049220957.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/Users/sathishkumarchandran/IdeaProjects/llm/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,8.138900,6.992688,0.036000,0.000000,0.034100,0.033800,20.000000
2,2.808500,1.317268,0.000000,0.000000,0.000000,0.000000,0.000000
3,2.060400,0.804722,0.000000,0.000000,0.000000,0.000000,0.000000


/Users/sathishkumarchandran/IdeaProjects/llm/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=60, training_loss=5.867596530914307, metrics={'train_runtime': 214.6415, 'train_samples_per_second': 1.118, 'train_steps_per_second': 0.28, 'total_flos': 32482032353280.0, 'train_loss': 5.867596530914307, 'epoch': 3.0})

In [39]:
# --- Step 6: Use the fine-tuned model for inference ---
from transformers import pipeline

# The trainer saves the model at the specified output directory
finetuned_model_path = "./t5-summarization-finetuned/checkpoint-60"
summarizer = pipeline("summarization", model=finetuned_model_path, tokenizer=tokenizer)

# Use the trained model to generate a new summary
new_feedback = "The project was consistently over budget and missed several key deadlines. The team, however, was highly diverse and collaborated effectively."
prompt = f"Instruction: Segment the following feedback into a summary. Input: {new_feedback}"

# Generate the summary
summary = summarizer(prompt, max_length=50, min_length=10, do_sample=False)

print(f"Original: {new_feedback}")
print(f"Generated Summary: {summary[0]['summary_text']}")

Device set to use cpu
Your max_length is set to 50, but your input_length is only 44. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original: The project was consistently over budget and missed several key deadlines. The team, however, was highly diverse and collaborated effectively.
Generated Summary: the project was consistently over budget and missed several key deadlines .


In [23]:
# !pip install evaluate
# !pip install nltk absl-py rouge-score


In [43]:
from transformers import pipeline

# The trainer saves the model at the specified output directory
finetuned_model_path = "./t5-summarization-finetuned/checkpoint-60"
summarizer = pipeline("summarization", model=finetuned_model_path, tokenizer=tokenizer)

new_feedback = "Availability has been a cornerstone in their services as well as an active and assertive response to our needs"
prompt = f"Instruction: Segment the following feedback into a summary. Input: {new_feedback}"

summary = summarizer(prompt, max_length=50, min_length=10, do_sample=False)

print(f"Original: {new_feedback}")
print(f"Generated Summary: {summary[0]['summary_text']}")


Device set to use cpu
Your max_length is set to 50, but your input_length is only 41. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=20)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original: Availability has been a cornerstone in their services as well as an active and assertive response to our needs
Generated Summary: Availability has been a cornerstone in their services and an active and assertive response to our needs .
